In [0]:
# ==========================================
# 04 - GOLD AGGREGATION
# Fraud Detection Lakehouse
# ==========================================

from pyspark.sql import functions as F

CATALOG_NAME = "fraud_detection"

SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"

spark.sql(f"USE CATALOG {CATALOG_NAME}")

print("Gold aggregation initialized")

In [0]:
# Load validated Silver tables

customers = spark.table(
    f"{CATALOG_NAME}.{SILVER_SCHEMA}.customers"
)

merchants = spark.table(
    f"{CATALOG_NAME}.{SILVER_SCHEMA}.merchants"
)

transactions = spark.table(
    f"{CATALOG_NAME}.{SILVER_SCHEMA}.transactions"
)

print("Customers:", customers.count())
print("Merchants:", merchants.count())
print("Transactions:", transactions.count())

In [0]:
# ==========================================
# CUSTOMER RISK METRICS
# ==========================================

customer_metrics = (
    transactions
    .groupBy("customer_id")
    .agg(
        F.count("transaction_id").alias("total_transactions"),

        F.sum("amount").alias("total_transaction_amount"),

        F.avg("amount").alias("average_transaction_amount"),

        F.sum("fraud_label").alias("fraudulent_transactions"),

        F.sum(
            F.when(F.col("amount") >= 50000, 1).otherwise(0)
        ).alias("high_value_transactions"),

        F.countDistinct("merchant_id").alias("unique_merchants"),

        F.min("transaction_timestamp").alias("first_transaction"),

        F.max("transaction_timestamp").alias("last_transaction")
    )
)

# Include customers with no transactions

customer_risk = (
    customers
    .join(customer_metrics, on="customer_id", how="left")
    .fillna({
        "total_transactions": 0,
        "total_transaction_amount": 0,
        "average_transaction_amount": 0,
        "fraudulent_transactions": 0,
        "high_value_transactions": 0,
        "unique_merchants": 0
    })
    .withColumn(
        "fraud_rate",
        F.when(
            F.col("total_transactions") > 0,
            F.col("fraudulent_transactions") /
            F.col("total_transactions")
        ).otherwise(0)
    )
)

# Rule-based risk categorization

customer_risk = customer_risk.withColumn(
    "risk_category",
    F.when(
        (F.col("fraudulent_transactions") >= 2) |
        (
            (F.col("fraudulent_transactions") >= 1) &
            (F.col("fraud_rate") >= 0.10)
        ),
        "HIGH"
    ).when(
        (F.col("fraudulent_transactions") == 1) |
        (F.col("high_value_transactions") >= 3),
        "MEDIUM"
    ).otherwise("LOW")
)

customer_risk = customer_risk.withColumn(
    "risk_assessment_timestamp",
    F.current_timestamp()
)

# Save as Gold Delta table

(
    customer_risk.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{CATALOG_NAME}.{GOLD_SCHEMA}.customer_risk_metrics"
    )
)

print("Customer risk metrics generated successfully!")

In [0]:
customer_risk_gold = spark.table(
    "fraud_detection.gold.customer_risk_metrics"
)

print("Total customers:", customer_risk_gold.count())

display(
    customer_risk_gold.limit(20)
)

In [0]:
# ==========================================
# FRAUD SUMMARY BY COUNTRY
# ==========================================

fraud_by_country = (
    transactions
    .groupBy("country")
    .agg(
        F.count("transaction_id").alias("total_transactions"),

        F.sum("fraud_label").alias("fraudulent_transactions"),

        F.sum("amount").alias("total_transaction_amount"),

        F.sum(
            F.when(F.col("fraud_label") == 1, F.col("amount"))
             .otherwise(0)
        ).alias("fraudulent_transaction_amount")
    )
    .withColumn(
        "fraud_rate",
        F.col("fraudulent_transactions") /
        F.col("total_transactions")
    )
    .withColumn(
        "aggregation_timestamp",
        F.current_timestamp()
    )
)

(
    fraud_by_country.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("fraud_detection.gold.fraud_summary_by_country")
)

print("Fraud summary by country generated successfully!")

In [0]:
country_gold = spark.table(
    "fraud_detection.gold.fraud_summary_by_country"
)

print("Countries:", country_gold.count())

display(country_gold.orderBy(F.desc("fraud_rate")))

In [0]:
# ==========================================
# FRAUD SUMMARY BY MERCHANT CATEGORY
# ==========================================

transaction_details = (
    transactions
    .join(
        merchants.select("merchant_id", "category"),
        on="merchant_id",
        how="left"
    )
)

fraud_by_category = (
    transaction_details
    .groupBy("category")
    .agg(
        F.count("transaction_id").alias("total_transactions"),

        F.sum("fraud_label").alias("fraudulent_transactions"),

        F.sum("amount").alias("total_transaction_amount"),

        F.avg("amount").alias("average_transaction_amount")
    )
    .withColumn(
        "fraud_rate",
        F.col("fraudulent_transactions") /
        F.col("total_transactions")
    )
    .withColumn(
        "aggregation_timestamp",
        F.current_timestamp()
    )
)

(
    fraud_by_category.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("fraud_detection.gold.fraud_summary_by_category")
)

print("Fraud summary by merchant category generated successfully!")

In [0]:
category_gold = spark.table(
    "fraud_detection.gold.fraud_summary_by_category"
)

print("Merchant categories:", category_gold.count())

display(
    category_gold.orderBy(F.desc("fraud_rate"))
)

In [0]:
# ==========================================
# DAILY TRANSACTION METRICS
# ==========================================

daily_metrics = (
    transactions
    .withColumn(
        "transaction_date",
        F.to_date("transaction_timestamp")
    )
    .groupBy("transaction_date")
    .agg(
        F.count("transaction_id").alias("total_transactions"),

        F.sum("fraud_label").alias("fraudulent_transactions"),

        F.sum("amount").alias("total_transaction_amount"),

        F.avg("amount").alias("average_transaction_amount"),

        F.sum(
            F.when(F.col("fraud_label") == 1, F.col("amount"))
             .otherwise(0)
        ).alias("fraudulent_transaction_amount")
    )
    .withColumn(
        "fraud_rate",
        F.col("fraudulent_transactions") /
        F.col("total_transactions")
    )
    .withColumn(
        "aggregation_timestamp",
        F.current_timestamp()
    )
)

(
    daily_metrics.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("fraud_detection.gold.daily_transaction_metrics")
)

print("Daily transaction metrics generated successfully!")

In [0]:
daily_gold = spark.table(
    "fraud_detection.gold.daily_transaction_metrics"
)

print("Total days:", daily_gold.count())

display(
    daily_gold.orderBy("transaction_date")
)

In [0]:
# ==========================================
# GOLD LAYER VALIDATION
# ==========================================

gold_tables = [
    "customer_risk_metrics",
    "fraud_summary_by_country",
    "fraud_summary_by_category",
    "daily_transaction_metrics"
]

for table_name in gold_tables:
    full_table_name = f"fraud_detection.gold.{table_name}"

    record_count = spark.table(full_table_name).count()

    print(f"{full_table_name}: {record_count:,} records")

In [0]:
customer_total = (
    spark.table("fraud_detection.gold.customer_risk_metrics")
    .agg(F.sum("total_transactions"))
    .first()[0]
)

silver_total = transactions.count()

print("Customer metrics transaction total:", customer_total)
print("Silver transaction total:", silver_total)

assert customer_total == silver_total, "Transaction totals do not match!"

print("Gold transaction reconciliation passed!")